In [1]:
# Import necessary libraries
import os
import cv2
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import img_to_array


In [2]:

# Configuration
IMG_SIZE = (128, 128)


In [3]:

# DataLoader class (same as provided)
class DataLoader:
    def __init__(self, data_path, img_size=(64, 64)):
        self.data_path = data_path
        self.img_size = img_size
        self.expected_classes = [chr(c) for c in range(65, 91)] + [str(i) for i in range(1, 10)]
        self.class_map = {i: self.expected_classes[i] for i in range(35)}
        
    def _edge_enhancement(self, img):
        blurred = cv2.GaussianBlur(img, (5,5), 2)
        thresholded = cv2.adaptiveThreshold(
            blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 11, 2
        )
        _, processed = cv2.threshold(
            thresholded, 70, 255, 
            cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU
        )
        return processed

    def load_dataset(self, test_size=0.2, validation_size=0.2):
        images, labels = [], []
        
        directories = [d for d in os.listdir(self.data_path) 
                       if os.path.isdir(os.path.join(self.data_path, d))]
        
        class_mapping = {class_name: idx for idx, class_name in enumerate(self.expected_classes)}
        
        for class_dir in directories:
            if class_dir not in class_mapping:
                print(f"Skipping unexpected directory: {class_dir}")
                continue
                
            class_idx = class_mapping[class_dir]
            class_path = os.path.join(self.data_path, class_dir)
            print(f"Processing class {class_dir} => Index {class_idx} ({class_idx+1}/35)")
            
            for img_file in os.listdir(class_path):
                img_path = os.path.join(class_path, img_file)
                img = cv2.imread(img_path, 0)  # Read as grayscale
                
                processed = self._edge_enhancement(img)
                resized = cv2.resize(processed, self.img_size)
                array_img = img_to_array(resized)
                
                images.append(array_img)
                labels.append(class_idx)
                
        images = np.array(images, dtype='float32') / 255.0
        labels = np.array(labels)
        
        X_train, X_test, y_train, y_test = train_test_split(
            images, labels, 
            test_size=test_size, 
            stratify=labels,
            random_state=42
        )
        
        X_train, X_val, y_train, y_val = train_test_split(
            X_train, y_train, 
            test_size=validation_size, 
            stratify=y_train,
            random_state=42
        )
        
        return X_train, X_val, X_test, y_train, y_val, y_test


In [4]:

# Load the dataset
DATA_PATH = '/kaggle/input/Indian'  # Update this path
loader = DataLoader(DATA_PATH, IMG_SIZE)
x_train, x_val, x_test, y_train, y_val, y_test = loader.load_dataset()


Processing class N => Index 13 (14/35)
Processing class 7 => Index 32 (33/35)
Processing class R => Index 17 (18/35)
Processing class 2 => Index 27 (28/35)
Processing class B => Index 1 (2/35)
Processing class I => Index 8 (9/35)
Processing class F => Index 5 (6/35)
Processing class H => Index 7 (8/35)
Processing class 5 => Index 30 (31/35)
Processing class E => Index 4 (5/35)
Processing class U => Index 20 (21/35)
Processing class M => Index 12 (13/35)
Processing class 8 => Index 33 (34/35)
Processing class X => Index 23 (24/35)
Processing class K => Index 10 (11/35)
Processing class Q => Index 16 (17/35)
Processing class Y => Index 24 (25/35)
Processing class S => Index 18 (19/35)
Processing class G => Index 6 (7/35)
Processing class A => Index 0 (1/35)
Processing class O => Index 14 (15/35)
Processing class T => Index 19 (20/35)
Processing class V => Index 21 (22/35)
Processing class Z => Index 25 (26/35)
Processing class 3 => Index 28 (29/35)
Processing class 1 => Index 26 (27/35)


In [5]:
# Flatten the images for SVM
x_train_flattened = x_train.reshape(x_train.shape[0], -1)
x_val_flattened = x_val.reshape(x_val.shape[0], -1)
x_test_flattened = x_test.reshape(x_test.shape[0], -1)

In [6]:
# Train the SVM model
svm_model = svm.SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(x_train_flattened, y_train)

SVC(kernel='linear', random_state=42)

In [7]:
# Evaluate the model on the validation set
y_val_pred = svm_model.predict(x_val_flattened)
print("Validation Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=list(loader.class_map.values())))
print("Validation Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))


Validation Classification Report:
              precision    recall  f1-score   support

           A       1.00      1.00      1.00       192
           B       1.00      1.00      1.00       192
           C       1.00      1.00      1.00       232
           D       1.00      1.00      1.00       192
           E       1.00      1.00      1.00       192
           F       1.00      1.00      1.00       192
           G       1.00      1.00      1.00       192
           H       1.00      1.00      1.00       192
           I       1.00      1.00      1.00       221
           J       1.00      1.00      1.00       192
           K       1.00      1.00      1.00       192
           L       1.00      1.00      1.00       192
           M       1.00      1.00      1.00       192
           N       1.00      1.00      1.00       192
           O       1.00      1.00      1.00       229
           P       1.00      1.00      1.00       192
           Q       1.00      1.00      1.00    

In [8]:
# Evaluate the model on the test set
y_test_pred = svm_model.predict(x_test_flattened)
print("Test Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=list(loader.class_map.values())))
print("Test Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Test Classification Report:
              precision    recall  f1-score   support

           A       1.00      1.00      1.00       240
           B       1.00      1.00      1.00       240
           C       1.00      1.00      1.00       289
           D       1.00      1.00      1.00       240
           E       1.00      1.00      1.00       240
           F       1.00      1.00      1.00       240
           G       1.00      1.00      1.00       240
           H       1.00      1.00      1.00       240
           I       1.00      1.00      1.00       276
           J       1.00      1.00      1.00       240
           K       1.00      1.00      1.00       240
           L       1.00      1.00      1.00       240
           M       1.00      1.00      1.00       240
           N       1.00      1.00      1.00       240
           O       1.00      1.00      1.00       286
           P       1.00      1.00      1.00       240
           Q       1.00      1.00      1.00       240